In [2]:
import sys
sys.path.append("..")

import yfinance as yf
import pandas as pd
from src.ingest.build import load_raw

dfs = load_raw()
series = {}
for nombre, df in zip(["X", "Y", "Z"], dfs):
    series[nombre] = df.set_index("Date")[f"Price_{nombre}"].sort_index().resample("ME").mean()

tickers = {"Brent": "BZ=F", "WTI": "CL=F", "Cobre": "HG=F", "Oro": "GC=F", "Acero": "SLX"}

for nombre, tk in tickers.items():
    try:
        ext = yf.download(tk, start="2010-01-01", end="2023-09-01", progress=False)["Close"]
        ext = ext.squeeze().resample("ME").mean()
        print(f"\n{nombre} ({tk}):")
        for s in ["X", "Y", "Z"]:
            comun = pd.concat([series[s], ext], axis=1, join="inner").dropna()
            comun.columns = ["serie", "ext"]
            niv = comun["serie"].corr(comun["ext"])
            ret = comun.pct_change().dropna().corr().iloc[0, 1]
            print(f"  {s}: niveles {niv:.2f} | retornos {ret:.2f}")
    except Exception as e:
        print(f"{nombre}: falló ({e})")


Brent (BZ=F):
  X: niveles 1.00 | retornos 1.00
  Y: niveles 0.50 | retornos 0.31
  Z: niveles 0.48 | retornos 0.45

WTI (CL=F):
  X: niveles 0.98 | retornos 0.89
  Y: niveles 0.53 | retornos 0.25
  Z: niveles 0.53 | retornos 0.36

Cobre (HG=F):
  X: niveles 0.65 | retornos 0.47
  Y: niveles 0.89 | retornos 0.53
  Z: niveles 0.87 | retornos 0.65

Oro (GC=F):
  X: niveles 0.24 | retornos 0.03
  Y: niveles 0.57 | retornos 0.12
  Z: niveles 0.50 | retornos 0.21

Acero (SLX):
  X: niveles 0.41 | retornos 0.62
  Y: niveles 0.78 | retornos 0.58
  Z: niveles 0.83 | retornos 0.54
